### Allscripts Sunrise (SCM) - Observation Hydration

**Source Tables:**
- _exponent._bronze_allscripts_scm_prod01_vw.dbo_cv3observationcur (observation records)
- _exponent._bronze_allscripts_scm_prod01_vw.dbo_cv3observationdocumentcur (links observations to documents, has RecordedDtm)
- _exponent._bronze_allscripts_scm_prod_01.dbo_cv3clientdocumentcur (links documents to patients via ClientGUID)

**Mapping Table:**
- _exponent.omop_mapping.domain_source_to_concept (observation concept mappings via ObsItemGUID / entry identifier)

**Join Path:**
1. cv3observationcur → cv3observationdocumentcur (via obs.GUID = obsdoc.ObservationGUID)
2. cv3observationdocumentcur → cv3clientdocumentcur (via obsdoc.OwnerGUID = doc.GUID)
3. cv3clientdocumentcur → source_to_person (via doc.ClientGUID)

**Strategy:**
- Use cv3observationcur for observation data (non-numeric values)
- Use RecordedDtm from cv3observationdocumentcur for observation date
- Map ObsItemGUID to OMOP observation concepts via mapping table
- Use UnitOfMeasure for unit information
- observation_type_concept_id = 32817 (EHR)

**Note:**
- Sunrise stores observations and measurements in cv3observationcur
- This notebook captures non-numeric observation types
- See allscripts_scm_measurement.ipynb for numeric values

In [0]:
%sql
-- TRUNCATE TABLE _exponent.omop_allscripts.observation

In [0]:
%sql
-- TRUNCATE TABLE _exponent.omop_scm.observation;


In [0]:
%sql
-- DELETE FROM _exponent.omop_silver.observation
-- WHERE source_system = 'allscripts_scm';


In [0]:
%sql
-- DELETE FROM _exponent.omop_mapping.source_to_observation
-- WHERE source_system = 'allscripts_scm';


In [0]:
source = 'allscripts_scm'

# Transformation

In [0]:
%sql
-- Create silver_observation temp view
-- Source: allscripts_scm
-- Join path: cv3observationcur → cv3observationdocumentcur → cv3clientdocumentcur → person

CREATE OR REPLACE TEMP VIEW silver_observation AS
WITH loinc_observation_map_ranked AS (
  SELECT
    source_concept.concept_code AS source_concept_code,
    source_concept.concept_id AS source_concept_id,
    COALESCE(
      mapped_standard.concept_id,
      CASE
        WHEN source_concept.standard_concept = 'S' AND source_concept.domain_id = 'Observation'
        THEN source_concept.concept_id
      END
    ) AS standard_concept_id,
    ROW_NUMBER() OVER (
      PARTITION BY source_concept.concept_code
      ORDER BY mapped_standard.concept_id DESC, source_concept.concept_id DESC
    ) AS rn
  FROM _exponent.omop.concept source_concept
  LEFT JOIN _exponent.omop.concept_relationship cr
    ON cr.concept_id_1 = source_concept.concept_id
   AND cr.relationship_id = 'Maps to'
   AND cr.invalid_reason IS NULL
  LEFT JOIN _exponent.omop.concept mapped_standard
    ON mapped_standard.concept_id = cr.concept_id_2
   AND mapped_standard.standard_concept = 'S'
   AND mapped_standard.domain_id = 'Observation'
   AND mapped_standard.invalid_reason IS NULL
  WHERE source_concept.vocabulary_id = 'LOINC'
    AND source_concept.invalid_reason IS NULL
), loinc_observation_map AS (
  SELECT source_concept_code, source_concept_id, standard_concept_id
  FROM loinc_observation_map_ranked
  WHERE rn = 1
    AND standard_concept_id IS NOT NULL
), raw_scm_observation_map AS (
  SELECT
    CAST(d.source_id AS STRING) AS obs_item_guid,
    LOWER(TRIM(CAST(d.source_value AS STRING))) AS source_value_clean,
    d.omop_concept_id AS source_concept_id,
    COALESCE(standard_concept.concept_id, mapped_standard.concept_id) AS standard_concept_id
  FROM _exponent.omop_mapping.domain_source_to_concept d
  LEFT JOIN _exponent.omop.concept mapped_standard
    ON mapped_standard.concept_id = d.omop_concept_id
   AND mapped_standard.standard_concept = 'S'
   AND mapped_standard.domain_id = 'Observation'
   AND mapped_standard.invalid_reason IS NULL
  LEFT JOIN _exponent.omop.concept_relationship cr
    ON cr.concept_id_1 = d.omop_concept_id
   AND cr.relationship_id = 'Maps to'
   AND cr.invalid_reason IS NULL
  LEFT JOIN _exponent.omop.concept standard_concept
    ON standard_concept.concept_id = cr.concept_id_2
   AND standard_concept.standard_concept = 'S'
   AND standard_concept.domain_id = 'Observation'
   AND standard_concept.invalid_reason IS NULL
  WHERE d.source_system = 'allscripts_scm'
    AND d.domain_id = 'Observation'
    AND d.active_flag = TRUE
), scm_observation_map_by_id AS (
  SELECT obs_item_guid, source_concept_id, standard_concept_id
  FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY obs_item_guid ORDER BY standard_concept_id DESC, source_concept_id DESC) AS rn
    FROM raw_scm_observation_map
    WHERE obs_item_guid IS NOT NULL
  )
  WHERE rn = 1
), scm_observation_map_by_value AS (
  SELECT source_value_clean, source_concept_id, standard_concept_id
  FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY source_value_clean ORDER BY standard_concept_id DESC, source_concept_id DESC) AS rn
    FROM raw_scm_observation_map
    WHERE source_value_clean IS NOT NULL AND source_value_clean <> ''
  )
  WHERE rn = 1
), catalog_standard_observation_map AS (
  SELECT
    LOWER(TRIM(CAST(concept_name AS STRING))) AS catalog_description_clean,
    concept_id AS source_concept_id,
    concept_id AS standard_concept_id
  FROM _exponent.omop.concept
  WHERE standard_concept = 'S'
    AND domain_id = 'Observation'
    AND invalid_reason IS NULL
), excluded_catalog_descriptions AS (
  SELECT explode(array(
    'outcome summary',
    'assessment',
    'subjective and objective box',
    'comments',
    'reason for admission',
    'goal for the day',
    'goal for the stay',
    'chief complaint quote',
    'nspromutinfoindividft_gen_a_nur',
    'objective statement',
    'nsicdxpastmedicalhx_gen_all_core_ft',
    'how patient addressed, profile',
    'nsicdxpastsurgicalhx_gen_all_core_ft',
    'problem selector problem 1',
    'problem selector plan 1',
    'problem selector problem 2',
    'clinical summary medical decision making free text box',
    'peripheral iv: insertion date',
    'caregiver name',
    'last bowel movement',
    'caregiver relation to patient',
    'caregiver address',
    'patient progress',
    'wet read launch ft',
    'problem selector plan 2',
    'problem selector problem 3',
    'patient portal link ft',
    'individualization/preferences',
    'nschartnoteft_gen_a_core',
    'caregiver phone number',
    'problem selector problem 4',
    'care plan',
    'time billing',
    'nstimeprovidercareinitiate_gen_er',
    'nscareinitiated _gen_er',
    'nspromutparticipcareft_gen_a_nur',
    'progress note details',
    'primary care provider',
    'procedure',
    'problem selector plan 4',
    'problem selector problem 5',
    'nsproptrightsupportname_gen_a_nur',
    'physical examination',
    'date and time:',
    'problem selector plan 5',
    'hospital course',
    'literature/videos given',
    'ventilator pm date due',
    'consult reason',
    'problem selector problem 6',
    'consult requested by name',
    'provider tokens',
    'care provider_api call',
    'consult requested date/time',
    'npi number (for sysadmin use only) :',
    'nsfollowupinstructions_ed_all_ed_ft',
    'final nursing electronic signature',
    'problem selector plan 6',
    'nsproptrightrepphone_gen_a_nur',
    'nspromutanxfearaddressft_gen_a_nur',
    'ns ed nurse reassess comment ft1',
    'financial assistance',
    'peripheral iv: removal date',
    'nsicdxfamilyhx_gen_all_core_ft',
    'discharge date/time',
    'nsdchc_medrecstatus_gen_all_core',
    'name of md/np/pa/do notified:'
  )) AS catalog_description_clean
), excluded_catalog_patterns AS (
  SELECT explode(array(
    '^problem selector (problem|plan) [0-9]+$',
    '^peripheral iv: .* date$',
    '^ns.*(_ft|ft[0-9]?|_gen_|_ed_|_all_)',
    '.*\\b(api call|tokens|signature|sysadmin|provider|caregiver|portal link|financial assistance)\\b.*',
    '.*\\b(note|summary|comments|statement|progress|course|instructions|reason|goal|plan|care plan|physical examination|date/time|time billing|literature/videos)\\b.*',
    '.*\\bconsult requested\\b.*'
  )) AS catalog_description_pattern
), staged AS (
  SELECT
    stp.person_id,
    COALESCE(
      obs_catalog_map.standard_concept_id,
      catalog_exact_map.standard_concept_id,
      entry_loinc.standard_concept_id,
      obs_id_map.standard_concept_id,
      obs_value_map.standard_concept_id,
      0
    ) AS observation_concept_id,
    CAST(obsdoc.RecordedDtm AS DATE) AS observation_date,
    obsdoc.RecordedDtm AS observation_datetime,
    32817 AS observation_type_concept_id,  -- EHR
    CASE
      WHEN obs.ValueText RLIKE '^-?[0-9]+\\.?[0-9]*$' THEN CAST(obs.ValueText AS DOUBLE)
      ELSE NULL
    END AS value_as_number,
    CASE
      WHEN obs.ValueText RLIKE '^-?[0-9]+\\.?[0-9]*$' THEN NULL
      ELSE obs.ValueText
    END AS value_as_string,
    NULL AS value_as_concept_id,
    NULL AS qualifier_concept_id,
    0 AS unit_concept_id,
    NULL AS provider_id,
    COALESCE(stvo.visit_occurrence_id, vo.visit_occurrence_id) AS visit_occurrence_id,
    NULL AS visit_detail_id,
    CONCAT('allscripts_scm', ' | ', obs.GUID) AS observation_source_value,
    COALESCE(
      obs_catalog_map.source_concept_id,
      catalog_exact_map.source_concept_id,
      entry_loinc.source_concept_id,
      obs_id_map.source_concept_id,
      obs_value_map.source_concept_id,
      0
    ) AS observation_source_concept_id,
    obs.UnitOfMeasure AS unit_source_value,
    NULL AS qualifier_source_value,
    obs.ValueText AS value_source_value,
    NULL AS observation_event_id,
    NULL AS obs_event_field_concept_id,
    'allscripts_scm' AS source_system,
    ROW_NUMBER() OVER (
      PARTITION BY CONCAT('allscripts_scm', ' | ', obs.GUID)
      ORDER BY obsdoc.RecordedDtm DESC, doc.GUID DESC
    ) AS rn
  FROM _exponent._bronze_allscripts_scm_prod01_vw.dbo_cv3observationcur obs
  INNER JOIN _exponent._bronze_allscripts_scm_prod01_vw.dbo_cv3observationdocumentcur obsdoc
    ON obs.GUID = obsdoc.ObservationGUID
  INNER JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_cv3clientdocumentcur doc
    ON obsdoc.OwnerGUID = doc.GUID
  LEFT JOIN _exponent._bronze_allscripts_scm.dbo_scaobservation sca
    ON CAST(sca.ObservationGUID AS DECIMAL(38,0)) = CAST(obs.GUID AS DECIMAL(38,0))
  LEFT JOIN _bronze.sca_acutecare_uat.dbo_scaobscatalogdim ocd
    ON ocd.ObsCatalogDimID = sca.ObsCatalogDimID
  INNER JOIN _exponent.omop_mapping.source_to_person stp
    ON CONCAT_WS(CHR(31), 'allscripts_scm', 'cv3client', 'GUID', CAST(doc.ClientGUID AS STRING)) = stp.person_source_value
    AND stp.active_flag = TRUE
  LEFT JOIN _exponent._bronze_allscripts_scm_prod01_vw.dbo_cv3observationentryitem entry_item
    ON entry_item.ObsItemGUID = obs.ObsItemGUID
  LEFT JOIN loinc_observation_map entry_loinc
    ON TRIM(CAST(entry_item.ObsEntryItemIdentifier AS STRING)) RLIKE '^[0-9]+-[0-9]+$'
   AND entry_loinc.source_concept_code = TRIM(CAST(entry_item.ObsEntryItemIdentifier AS STRING))
  LEFT JOIN scm_observation_map_by_id obs_id_map
    ON obs_id_map.obs_item_guid = CAST(obs.ObsItemGUID AS STRING)
  LEFT JOIN scm_observation_map_by_value obs_value_map
    ON obs_value_map.source_value_clean = LOWER(TRIM(CAST(entry_item.ObsEntryItemIdentifier AS STRING)))
  LEFT JOIN scm_observation_map_by_value obs_catalog_map
    ON obs_catalog_map.source_value_clean = LOWER(TRIM(CAST(ocd.Description AS STRING)))
  LEFT JOIN catalog_standard_observation_map catalog_exact_map
    ON catalog_exact_map.catalog_description_clean = LOWER(TRIM(CAST(ocd.Description AS STRING)))
  LEFT JOIN excluded_catalog_descriptions excluded_catalog
    ON excluded_catalog.catalog_description_clean = LOWER(TRIM(CAST(ocd.Description AS STRING)))
  LEFT JOIN excluded_catalog_patterns excluded_pattern
    ON LOWER(TRIM(CAST(ocd.Description AS STRING))) RLIKE excluded_pattern.catalog_description_pattern
  LEFT JOIN _exponent.omop_mapping.source_to_visit_occurrence stvo
    ON stvo.visit_occurrence_source_value = CONCAT_WS(CHR(31), 'allscripts_scm', 'dbo_cv3clientvisit', 'GUID', CAST(doc.ClientVisitGUID AS STRING))
   AND stvo.source_system = 'allscripts_scm'
   AND stvo.active_flag = TRUE
  LEFT JOIN _exponent.omop_scm.visit_occurrence vo
    ON vo.visit_source_value = CONCAT_WS(CHR(31), 'allscripts_scm', 'dbo_cv3clientvisit', 'GUID', CAST(doc.ClientVisitGUID AS STRING))
  WHERE obs.GUID IS NOT NULL
    AND obs.StatusType = 1  -- Performed
    AND obs.ValueText IS NOT NULL
    AND obsdoc.RecordedDtm >= TIMESTAMP('1900-01-01')
    AND obsdoc.RecordedDtm <= CURRENT_TIMESTAMP()
    AND NOT (obs.ValueText RLIKE '^-?[0-9]+\\.?[0-9]*$')  -- Non-numeric values only
    AND excluded_catalog.catalog_description_clean IS NULL
    AND excluded_pattern.catalog_description_pattern IS NULL
)
SELECT
  person_id,
  observation_concept_id,
  observation_date,
  observation_datetime,
  observation_type_concept_id,
  value_as_number,
  value_as_string,
  value_as_concept_id,
  qualifier_concept_id,
  unit_concept_id,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  observation_source_value,
  observation_source_concept_id,
  unit_source_value,
  qualifier_source_value,
  value_source_value,
  observation_event_id,
  obs_event_field_concept_id,
  source_system
FROM staged
WHERE rn = 1
-- LIMIT 10000




# Merge to Silver

In [0]:
%sql
MERGE INTO _exponent.omop_silver.observation AS t
USING silver_observation AS s
ON t.observation_source_value = s.observation_source_value

WHEN MATCHED AND (
     NOT (t.person_id <=> s.person_id)
  OR NOT (t.observation_concept_id <=> s.observation_concept_id)
  OR NOT (t.observation_date <=> s.observation_date)
  OR NOT (t.observation_datetime <=> s.observation_datetime)
  OR NOT (t.observation_type_concept_id <=> s.observation_type_concept_id)
  OR NOT (t.value_as_number <=> s.value_as_number)
  OR NOT (t.value_as_string <=> s.value_as_string)
  OR NOT (t.value_as_concept_id <=> s.value_as_concept_id)
  OR NOT (t.qualifier_concept_id <=> s.qualifier_concept_id)
  OR NOT (t.unit_concept_id <=> s.unit_concept_id)
  OR NOT (t.provider_id <=> s.provider_id)
  OR NOT (t.visit_occurrence_id <=> s.visit_occurrence_id)
  OR NOT (t.visit_detail_id <=> s.visit_detail_id)
  OR NOT (t.observation_source_concept_id <=> s.observation_source_concept_id)
  OR NOT (t.unit_source_value <=> s.unit_source_value)
  OR NOT (t.qualifier_source_value <=> s.qualifier_source_value)
  OR NOT (t.value_source_value <=> s.value_source_value)
  OR NOT (t.observation_event_id <=> s.observation_event_id)
  OR NOT (t.obs_event_field_concept_id <=> s.obs_event_field_concept_id)
  OR NOT (t.source_system <=> s.source_system)
)
THEN UPDATE SET
  t.person_id                      = s.person_id,
  t.observation_concept_id         = s.observation_concept_id,
  t.observation_date               = s.observation_date,
  t.observation_datetime           = s.observation_datetime,
  t.observation_type_concept_id    = s.observation_type_concept_id,
  t.value_as_number                = s.value_as_number,
  t.value_as_string                = s.value_as_string,
  t.value_as_concept_id            = s.value_as_concept_id,
  t.qualifier_concept_id           = s.qualifier_concept_id,
  t.unit_concept_id                = s.unit_concept_id,
  t.provider_id                    = s.provider_id,
  t.visit_occurrence_id            = s.visit_occurrence_id,
  t.visit_detail_id                = s.visit_detail_id,
  t.observation_source_concept_id  = s.observation_source_concept_id,
  t.unit_source_value              = s.unit_source_value,
  t.qualifier_source_value         = s.qualifier_source_value,
  t.value_source_value             = s.value_source_value,
  t.observation_event_id           = s.observation_event_id,
  t.obs_event_field_concept_id     = s.obs_event_field_concept_id,
  t.source_system                  = s.source_system

WHEN NOT MATCHED THEN INSERT (
  person_id,
  observation_concept_id,
  observation_date,
  observation_datetime,
  observation_type_concept_id,
  value_as_number,
  value_as_string,
  value_as_concept_id,
  qualifier_concept_id,
  unit_concept_id,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  observation_source_value,
  observation_source_concept_id,
  unit_source_value,
  qualifier_source_value,
  value_source_value,
  observation_event_id,
  obs_event_field_concept_id,
  source_system
)
VALUES (
  s.person_id,
  s.observation_concept_id,
  s.observation_date,
  s.observation_datetime,
  s.observation_type_concept_id,
  s.value_as_number,
  s.value_as_string,
  s.value_as_concept_id,
  s.qualifier_concept_id,
  s.unit_concept_id,
  s.provider_id,
  s.visit_occurrence_id,
  s.visit_detail_id,
  s.observation_source_value,
  s.observation_source_concept_id,
  s.unit_source_value,
  s.qualifier_source_value,
  s.value_source_value,
  s.observation_event_id,
  s.obs_event_field_concept_id,
  s.source_system
);

# Populate Mapping Table

In [0]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_observation (
    source_system,
    observation_source_value,
    person_id,
    active_flag,
    created_tsp,
    last_mod_tsp,
    merge_id,
    merge_reason
)
SELECT
    s.source_system,
    s.observation_source_value,
    s.person_id,
    TRUE AS active_flag,
    current_timestamp() AS created_tsp,
    current_timestamp() AS last_mod_tsp,
    NULL AS merge_id,
    NULL AS merge_reason
FROM (
    SELECT
        source_system,
        observation_source_value,
        person_id
    FROM (
        SELECT
            source_system,
            observation_source_value,
            person_id,
            ROW_NUMBER() OVER (
                PARTITION BY observation_source_value
                ORDER BY person_id
            ) AS rn
        FROM _exponent.omop_silver.observation
        WHERE source_system = 'allscripts_scm'
    ) deduped
    WHERE rn = 1
) s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_observation x
  ON s.observation_source_value = x.observation_source_value
 AND x.source_system = 'allscripts_scm';

# Merge to Gold

In [0]:
%sql
MERGE INTO _exponent.omop_scm.observation AS gold_obs
USING (
  WITH active_mapping AS (
    SELECT
      observation_id,
      observation_source_value,
      ROW_NUMBER() OVER (
        PARTITION BY observation_source_value
        ORDER BY last_mod_tsp DESC, observation_id DESC
      ) AS rn
    FROM _exponent.omop_mapping.source_to_observation
    WHERE source_system = 'allscripts_scm'
      AND active_flag = TRUE
  ), src_dedup AS (
    SELECT
      m.observation_id,
      s.person_id,
      s.observation_concept_id,
      s.observation_date,
      s.observation_datetime,
      s.observation_type_concept_id,
      s.value_as_number,
      s.value_as_string,
      s.value_as_concept_id,
      s.qualifier_concept_id,
      s.unit_concept_id,
      s.provider_id,
      s.visit_occurrence_id,
      s.visit_detail_id,
      s.observation_source_value,
      s.observation_source_concept_id,
      s.unit_source_value,
      s.qualifier_source_value,
      s.value_source_value,
      s.observation_event_id,
      s.obs_event_field_concept_id,
      ROW_NUMBER() OVER (
        PARTITION BY m.observation_id
        ORDER BY s.observation_datetime DESC, s.person_id DESC
      ) AS rn
    FROM _exponent.omop_silver.observation s
    JOIN _exponent.omop_scm.person p
      ON p.person_id = s.person_id
    JOIN active_mapping m
      ON m.observation_source_value = s.observation_source_value
     AND m.rn = 1
    WHERE s.source_system = 'allscripts_scm'
  )
  SELECT
    observation_id,
    person_id,
    observation_concept_id,
    observation_date,
    observation_datetime,
    observation_type_concept_id,
    value_as_number,
    value_as_string,
    value_as_concept_id,
    qualifier_concept_id,
    unit_concept_id,
    provider_id,
    visit_occurrence_id,
    visit_detail_id,
    observation_source_value,
    observation_source_concept_id,
    unit_source_value,
    qualifier_source_value,
    value_source_value,
    observation_event_id,
    obs_event_field_concept_id
  FROM src_dedup
  WHERE rn = 1
) AS src
ON gold_obs.observation_id = src.observation_id

WHEN MATCHED AND NOT (
  gold_obs.person_id <=> src.person_id
  AND gold_obs.observation_concept_id <=> src.observation_concept_id
  AND gold_obs.observation_date <=> src.observation_date
  AND gold_obs.observation_datetime <=> src.observation_datetime
  AND gold_obs.observation_type_concept_id <=> src.observation_type_concept_id
  AND gold_obs.value_as_number <=> src.value_as_number
  AND gold_obs.value_as_string <=> src.value_as_string
  AND gold_obs.value_as_concept_id <=> src.value_as_concept_id
  AND gold_obs.qualifier_concept_id <=> src.qualifier_concept_id
  AND gold_obs.unit_concept_id <=> src.unit_concept_id
  AND gold_obs.provider_id <=> src.provider_id
  AND gold_obs.visit_occurrence_id <=> src.visit_occurrence_id
  AND gold_obs.visit_detail_id <=> src.visit_detail_id
  AND gold_obs.observation_source_value <=> src.observation_source_value
  AND gold_obs.observation_source_concept_id <=> src.observation_source_concept_id
  AND gold_obs.unit_source_value <=> src.unit_source_value
  AND gold_obs.qualifier_source_value <=> src.qualifier_source_value
  AND gold_obs.value_source_value <=> src.value_source_value
  AND gold_obs.observation_event_id <=> src.observation_event_id
  AND gold_obs.obs_event_field_concept_id <=> src.obs_event_field_concept_id
)
THEN UPDATE SET
  gold_obs.person_id                     = src.person_id,
  gold_obs.observation_concept_id        = src.observation_concept_id,
  gold_obs.observation_date              = src.observation_date,
  gold_obs.observation_datetime          = src.observation_datetime,
  gold_obs.observation_type_concept_id   = src.observation_type_concept_id,
  gold_obs.value_as_number               = src.value_as_number,
  gold_obs.value_as_string               = src.value_as_string,
  gold_obs.value_as_concept_id           = src.value_as_concept_id,
  gold_obs.qualifier_concept_id          = src.qualifier_concept_id,
  gold_obs.unit_concept_id               = src.unit_concept_id,
  gold_obs.provider_id                   = src.provider_id,
  gold_obs.visit_occurrence_id           = src.visit_occurrence_id,
  gold_obs.visit_detail_id               = src.visit_detail_id,
  gold_obs.observation_source_value      = src.observation_source_value,
  gold_obs.observation_source_concept_id = src.observation_source_concept_id,
  gold_obs.unit_source_value             = src.unit_source_value,
  gold_obs.qualifier_source_value        = src.qualifier_source_value,
  gold_obs.value_source_value            = src.value_source_value,
  gold_obs.observation_event_id          = src.observation_event_id,
  gold_obs.obs_event_field_concept_id    = src.obs_event_field_concept_id

WHEN NOT MATCHED THEN INSERT (
  observation_id,
  person_id,
  observation_concept_id,
  observation_date,
  observation_datetime,
  observation_type_concept_id,
  value_as_number,
  value_as_string,
  value_as_concept_id,
  qualifier_concept_id,
  unit_concept_id,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  observation_source_value,
  observation_source_concept_id,
  unit_source_value,
  qualifier_source_value,
  value_source_value,
  observation_event_id,
  obs_event_field_concept_id
)
VALUES (
  src.observation_id,
  src.person_id,
  src.observation_concept_id,
  src.observation_date,
  src.observation_datetime,
  src.observation_type_concept_id,
  src.value_as_number,
  src.value_as_string,
  src.value_as_concept_id,
  src.qualifier_concept_id,
  src.unit_concept_id,
  src.provider_id,
  src.visit_occurrence_id,
  src.visit_detail_id,
  src.observation_source_value,
  src.observation_source_concept_id,
  src.unit_source_value,
  src.qualifier_source_value,
  src.value_source_value,
  src.observation_event_id,
  src.obs_event_field_concept_id
);

In [0]:
%sql
MERGE INTO _exponent.omop_allscripts.observation AS gold_obs
USING (
  WITH active_mapping AS (
    SELECT
      observation_id,
      observation_source_value,
      ROW_NUMBER() OVER (
        PARTITION BY observation_source_value
        ORDER BY last_mod_tsp DESC, observation_id DESC
      ) AS rn
    FROM _exponent.omop_mapping.source_to_observation
    WHERE source_system = 'allscripts_scm'
      AND active_flag = TRUE
  ), src_dedup AS (
    SELECT
      m.observation_id,
      s.person_id,
      s.observation_concept_id,
      s.observation_date,
      s.observation_datetime,
      s.observation_type_concept_id,
      s.value_as_number,
      s.value_as_string,
      s.value_as_concept_id,
      s.qualifier_concept_id,
      s.unit_concept_id,
      s.provider_id,
      s.visit_occurrence_id,
      s.visit_detail_id,
      s.observation_source_value,
      s.observation_source_concept_id,
      s.unit_source_value,
      s.qualifier_source_value,
      s.value_source_value,
      s.observation_event_id,
      s.obs_event_field_concept_id,
      ROW_NUMBER() OVER (
        PARTITION BY m.observation_id
        ORDER BY s.observation_datetime DESC, s.person_id DESC
      ) AS rn
    FROM _exponent.omop_silver.observation s
    JOIN _exponent.omop_scm.person p
      ON p.person_id = s.person_id
    JOIN active_mapping m
      ON m.observation_source_value = s.observation_source_value
     AND m.rn = 1
    WHERE s.source_system = 'allscripts_scm'
  )
  SELECT
    observation_id,
    person_id,
    observation_concept_id,
    observation_date,
    observation_datetime,
    observation_type_concept_id,
    value_as_number,
    value_as_string,
    value_as_concept_id,
    qualifier_concept_id,
    unit_concept_id,
    provider_id,
    visit_occurrence_id,
    visit_detail_id,
    observation_source_value,
    observation_source_concept_id,
    unit_source_value,
    qualifier_source_value,
    value_source_value,
    observation_event_id,
    obs_event_field_concept_id
  FROM src_dedup
  WHERE rn = 1
) AS src
ON gold_obs.observation_id = src.observation_id

WHEN MATCHED AND NOT (
  gold_obs.person_id <=> src.person_id
  AND gold_obs.observation_concept_id <=> src.observation_concept_id
  AND gold_obs.observation_date <=> src.observation_date
  AND gold_obs.observation_datetime <=> src.observation_datetime
  AND gold_obs.observation_type_concept_id <=> src.observation_type_concept_id
  AND gold_obs.value_as_number <=> src.value_as_number
  AND gold_obs.value_as_string <=> src.value_as_string
  AND gold_obs.value_as_concept_id <=> src.value_as_concept_id
  AND gold_obs.qualifier_concept_id <=> src.qualifier_concept_id
  AND gold_obs.unit_concept_id <=> src.unit_concept_id
  AND gold_obs.provider_id <=> src.provider_id
  AND gold_obs.visit_occurrence_id <=> src.visit_occurrence_id
  AND gold_obs.visit_detail_id <=> src.visit_detail_id
  AND gold_obs.observation_source_value <=> src.observation_source_value
  AND gold_obs.observation_source_concept_id <=> src.observation_source_concept_id
  AND gold_obs.unit_source_value <=> src.unit_source_value
  AND gold_obs.qualifier_source_value <=> src.qualifier_source_value
  AND gold_obs.value_source_value <=> src.value_source_value
  AND gold_obs.observation_event_id <=> src.observation_event_id
  AND gold_obs.obs_event_field_concept_id <=> src.obs_event_field_concept_id
)
THEN UPDATE SET
  gold_obs.person_id                     = src.person_id,
  gold_obs.observation_concept_id        = src.observation_concept_id,
  gold_obs.observation_date              = src.observation_date,
  gold_obs.observation_datetime          = src.observation_datetime,
  gold_obs.observation_type_concept_id   = src.observation_type_concept_id,
  gold_obs.value_as_number               = src.value_as_number,
  gold_obs.value_as_string               = src.value_as_string,
  gold_obs.value_as_concept_id           = src.value_as_concept_id,
  gold_obs.qualifier_concept_id          = src.qualifier_concept_id,
  gold_obs.unit_concept_id               = src.unit_concept_id,
  gold_obs.provider_id                   = src.provider_id,
  gold_obs.visit_occurrence_id           = src.visit_occurrence_id,
  gold_obs.visit_detail_id               = src.visit_detail_id,
  gold_obs.observation_source_value      = src.observation_source_value,
  gold_obs.observation_source_concept_id = src.observation_source_concept_id,
  gold_obs.unit_source_value             = src.unit_source_value,
  gold_obs.qualifier_source_value        = src.qualifier_source_value,
  gold_obs.value_source_value            = src.value_source_value,
  gold_obs.observation_event_id          = src.observation_event_id,
  gold_obs.obs_event_field_concept_id    = src.obs_event_field_concept_id

WHEN NOT MATCHED THEN INSERT (
  observation_id,
  person_id,
  observation_concept_id,
  observation_date,
  observation_datetime,
  observation_type_concept_id,
  value_as_number,
  value_as_string,
  value_as_concept_id,
  qualifier_concept_id,
  unit_concept_id,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  observation_source_value,
  observation_source_concept_id,
  unit_source_value,
  qualifier_source_value,
  value_source_value,
  observation_event_id,
  obs_event_field_concept_id
)
VALUES (
  src.observation_id,
  src.person_id,
  src.observation_concept_id,
  src.observation_date,
  src.observation_datetime,
  src.observation_type_concept_id,
  src.value_as_number,
  src.value_as_string,
  src.value_as_concept_id,
  src.qualifier_concept_id,
  src.unit_concept_id,
  src.provider_id,
  src.visit_occurrence_id,
  src.visit_detail_id,
  src.observation_source_value,
  src.observation_source_concept_id,
  src.unit_source_value,
  src.qualifier_source_value,
  src.value_source_value,
  src.observation_event_id,
  src.obs_event_field_concept_id
);

# Mapping QA


In [0]:
%sql
SELECT
  COUNT(*) AS total_rows,
  SUM(CASE WHEN observation_concept_id = 0 THEN 1 ELSE 0 END) AS unmapped_rows,
  ROUND(100.0 * SUM(CASE WHEN observation_concept_id = 0 THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0), 2) AS pct_unmapped
FROM _exponent.omop_scm.observation;
